# Thesis pipeline demonstration

**Development note:** If `participant_labels.csv` is absent, labels come from augmented/Kaggle-style metadata (e.g. `Data_AUG_*.csv`). Use official ADReSS `participant_labels.csv` for final thesis evaluation. See `README.md` and `src/data_loading.py`.

This notebook walks through the same stages as `run_pipeline.py` for supervisor review: dataset discovery, preprocessing, linguistic features, baselines, shuffled-label sanity check, optimization, SHAP, stability metrics, and trade-off figures.

**Run all cells from the project root** `C:\Users\Yazan\Thesis` (Kernel working directory should be the project root so `import src` resolves).

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Project root:", ROOT)

Project root: C:\Users\Yazan\Thesis


In [2]:
from src.utils import setup_logging
from src import config
from src.data_loading import (
    load_label_table,
    print_directory_tree,
    summarize_cohort,
    validate_paths,
)

setup_logging()
print_directory_tree(config.DATA_DIR, max_depth=2, max_files=50)
validate_paths()


--- Discovered dataset root: C:\Users\Yazan\Thesis\Data ---
[dir] .
  Data_AUG_13.11.2024_output.csv
  Mina Testing 50transcripts - Mina Testing 50transcripts.csv
  Training_No.csv
  df_test.csv
  df_train.csv
  svm_model.pkl
[dir] Archive
[dir] Archive\Process-rec-001
  Process-rec-001__CTD.txt
  Process-rec-001__CTD.wav
  Process-rec-001__PFT.txt
  Process-rec-001__PFT.wav
  Process-rec-001__SFT.txt
  Process-rec-001__SFT.wav
[dir] Archive\Process-rec-002
  Process-rec-002__CTD.txt
  Process-rec-002__CTD.wav
  Process-rec-002__PFT.txt
  Process-rec-002__PFT.wav
  Process-rec-002__SFT.txt
  Process-rec-002__SFT.wav
[dir] Archive\Process-rec-003
  Process-rec-003__CTD.txt
  Process-rec-003__CTD.wav
  Process-rec-003__PFT.txt
  Process-rec-003__PFT.wav
  Process-rec-003__SFT.txt
  Process-rec-003__SFT.wav
[dir] Archive\Process-rec-004
  Process-rec-004__CTD.txt
  Process-rec-004__CTD.wav
  Process-rec-004__PFT.txt
  Process-rec-004__PFT.wav
  Process-rec-004__SFT.txt
  Process-rec-004_

{'project_root': 'C:\\Users\\Yazan\\Thesis',
 'data_dir': 'C:\\Users\\Yazan\\Thesis\\Data',
 'has_archive': True,
 'df_train': WindowsPath('C:/Users/Yazan/Thesis/Data/df_train.csv'),
 'augmented_csv': WindowsPath('C:/Users/Yazan/Thesis/Data/Data_AUG_13.11.2024_output.csv'),
 'has_participant_labels_csv': False,
 'label_source': 'Data_AUG*.csv (development)'}

In [3]:
labels = load_label_table()
summarize_cohort(labels)
labels.head()

2026-05-18 10:42:18,481 | WARNING | src.data_loading | LABEL_MODE=binary_hc_dementia: dropped 59 rows not in {HC, Dementia} (e.g. MCI).
2026-05-18 10:42:18,482 | INFO | src.data_loading | Loaded labels from C:\Users\Yazan\Thesis\Data\Data_AUG_13.11.2024_output.csv (98 rows)
2026-05-18 10:42:18,484 | WARNING | src.data_loading | DEVELOPMENT labels: using augmented/Kaggle-style metadata at C:\Users\Yazan\Thesis\Data\Data_AUG_13.11.2024_output.csv (THESIS_LABEL_MODE=binary_hc_dementia). This is **not** the official ADReSS AD vs CN protocol. TODO(THESIS): add C:\Users\Yazan\Thesis\Data\participant_labels.csv with official participant IDs and labels before final experiments.

--- Label summary ---
diagnosis_label
0    82
1    16


,participant_id,diagnosis_label
0,Process-rec-006,1
1,Process-rec-007,0
2,Process-rec-009,0
3,Process-rec-010,0
4,Process-rec-012,0


In [4]:
from src.preprocessing import build_cleaned_corpus, run_quality_checks

corpus = build_cleaned_corpus(labels["participant_id"].astype(str).tolist())
run_quality_checks(corpus)
corpus[["participant_id", "word_count", "cleaned_transcript"]].head(3)

,participant_id,word_count,cleaned_transcript
0,Process-rec-006,118,fa a family in the kitchen mother washing up b...
1,Process-rec-007,204,washing dishes and drying them the sink is ove...
2,Process-rec-009,170,the picture is a kitchen in a house there's a ...


In [5]:
from src.feature_extraction import extract_features_table

features = extract_features_table(corpus)
table = features.merge(labels, on="participant_id", how="inner")
table.head()

ModuleNotFoundError: No module named 'spacy'

In [ ]:
from pathlib import Path
from src.config import OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
feat_path = OUTPUT_DIR / "features.csv"
table.to_csv(feat_path, index=False)
print("Saved", feat_path)

In [ ]:
import pandas as pd
from src.modeling import run_baseline_suite, run_shuffled_label_suite

X = table.copy()
y = table["diagnosis_label"].values.astype(int)

baseline = run_baseline_suite(X, y)
baseline

In [ ]:
shuf = run_shuffled_label_suite(X, y)
shuf

In [ ]:
from src.optimization import run_optimization_suite

opt_df, opt_models = run_optimization_suite(X, y)
opt_df

In [ ]:
from sklearn.base import clone
from src.modeling import get_named_baselines
from src.explainability import compute_shap_suite

baseline_fitted = {}
for name, tmpl in get_named_baselines().items():
    m = clone(tmpl)
    m.fit(X[list(config.FEATURE_COLUMNS)], y)
    baseline_fitted[name] = m

shap_b = compute_shap_suite(baseline_fitted, X, variant="baseline")
shap_o = compute_shap_suite(opt_models, X, variant="optimized")
shap_b.head()

In [ ]:
from src.stability import (
    run_stability_analysis,
    compare_baseline_vs_optimized_global,
    composite_stability_score,
)
import pandas as pd

stab = pd.concat(
    [
        run_stability_analysis(X, y, skip_svm_shap=True),
        compare_baseline_vs_optimized_global(shap_b, shap_o),
    ],
    ignore_index=True,
)
stab_core = stab[
    stab["metric"].isin(
        [
            "spearman_rank_importance",
            "jaccard_top5_features",
            "mean_cv_shap_magnitude_across_features",
        ]
    )
]
stab_sum = composite_stability_score(stab_core)
stab.head(10), stab_sum

In [ ]:
from src.visualization import (
    plot_performance_comparison,
    plot_stability_comparison,
    plot_tradeoff_scatter,
)
from src.config import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_performance_comparison(baseline, opt_df, FIGURES_DIR / "performance_comparison.png")
plot_stability_comparison(
    stab[stab["metric"] == "spearman_rank_importance"],
    FIGURES_DIR / "stability_comparison.png",
)
perf = pd.concat(
    [
        baseline[["model", "roc_auc_mean"]].assign(kind="baseline"),
        opt_df[["model", "roc_auc_mean"]].assign(kind="optimized"),
    ],
    ignore_index=True,
)
plot_tradeoff_scatter(perf, stab_sum, FIGURES_DIR / "tradeoff_auroc_stability.png")
print("Figures written to", FIGURES_DIR)